# Eventi estremi 2019 — qualità della previsione evento per evento

Gli eventi non sono scelti dal calendario: sono rilevati dallo score MTGFlow,
raggruppando le ore in cui molte località stanno contemporaneamente nella coda
estrema dell'anno. Episodi vicini sono fusi, perché un'ondata di calore produce
un episodio al giorno invece che uno continuo.

Per ogni evento si guardano le stesse metriche dei notebook precedenti, separate
per fascia di produzione: **MAE, RMSE, PICP, NMPIL, CLC**, più bias con segno,
boxplot degli errori assoluti e della NMPIL per riga, e istogrammi dell'errore.
Lo strato di riferimento è `normal`, cioè le righe non anomale dell'intero 2019.

Tutte le righe diurne valide, nessun campionamento.

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.anomaly_driver as anomaly_driver
import physiq_pv.reporting.anomaly_extremes as anomaly_extremes
import physiq_pv.reporting.report_dump as report_dump

anomaly_driver = importlib.reload(anomaly_driver)
anomaly_extremes = importlib.reload(anomaly_extremes)
report_dump = importlib.reload(report_dump)

RUN_NAME = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_seed1'
DETECTOR_SEED = 15

out_dir = ROOT / 'outputs' / RUN_NAME
scores_path = (ROOT / 'outputs' / 'pvgis_mtgflow' / 'downstream_dense'
               / f'seed_{DETECTOR_SEED}' / 'anomaly_scores.csv')
labels_path = out_dir / anomaly_driver.DRIVER_LABELS_FILE

for required in (scores_path, out_dir / 'predictions.csv',
                 out_dir / 'reference_production_peaks.csv'):
    if not required.is_file():
        raise FileNotFoundError(required)

driver_labels = (
    anomaly_driver.load_driver_labels(labels_path) if labels_path.is_file() else None
)
print('Run      :', out_dir)
print('Score    :', scores_path)
print('Etichette driver:', 'caricate' if driver_labels is not None else 'assenti')

## 1. Rilevamento eventi

`EXTREME_QUANTILE` definisce cosa è "coda", `MERGE_GAP_HOURS` quanto lontani
possono stare due episodi per essere lo stesso evento. `MIN_SHARE='auto'` legge
la quota dalla distribuzione oraria: l'1% di ore più intense dell'anno.

Se escono pochi eventi, abbassare `EXTREME_QUANTILE`: il taglio è un quantile
della distribuzione degli score, quindi quando gli eventi occupano una frazione
non trascurabile dell'anno il taglio finisce *dentro* la loro popolazione e ne
seleziona solo il più intenso. Con `0.99` la coda è dieci volte più larga.

In [ ]:
EXTREME_QUANTILE = 0.999
MERGE_GAP_HOURS = 48
MIN_DURATION_HOURS = 3
MAX_GAP_HOURS = 6

series = anomaly_extremes.regional_extreme_series(scores_path, quantile=EXTREME_QUANTILE)
episodes = anomaly_extremes.detect_extreme_episodes(
    series, min_share='auto', max_gap_hours=MAX_GAP_HOURS,
    min_duration_hours=MIN_DURATION_HOURS,
)
events = anomaly_extremes.group_episodes_into_events(
    episodes, merge_gap_hours=MERGE_GAP_HOURS
)

print(f"Taglio coda      : {series.attrs['cut']:.1f}")
print(f"Quota minima     : {episodes.attrs['min_share']:.5f}")
print(f"Episodi rilevati : {len(episodes)}")
print(f"EVENTI RILEVATI  : {len(events)}")
display(series['extreme_share'].describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_frame().round(5))

if events.empty:
    raise ValueError(
        'Nessun evento rilevato: abbassare EXTREME_QUANTILE (0.99) oppure '
        'MIN_DURATION_HOURS. Le celle successive non hanno nulla da confrontare.'
    )
display(events.drop(columns='days'))

In [ ]:
# Quanti giorni copre ogni evento e con quale composizione per driver.
if not events.empty:
    summary = []
    for row in events.itertuples(index=False):
        entry = {
            'evento': row.event,
            'giorni': ' '.join(row.days),
            'n_giorni': len(row.days),
            'durata_h': row.duration_hours,
            'picco': round(row.peak_score, 1),
            'finestre_estreme': row.n_extreme_windows,
        }
        if driver_labels is not None:
            stamps = pd.to_datetime(driver_labels['timestamp'])
            window = driver_labels[
                (stamps >= pd.Timestamp(row.start).normalize())
                & (stamps <= pd.Timestamp(row.end).normalize() + pd.Timedelta(hours=23))
            ]
            mix = window['driver'].value_counts(normalize=True)
            entry.update({f'share_{name}': round(mix.get(name, 0.0), 3)
                          for name in ('solar', 'temperature', 'wind')})
            entry['driver'] = mix.idxmax() if len(mix) else ''
        summary.append(entry)
    event_summary = pd.DataFrame(summary)
    display(event_summary)

## 2. Metriche per evento e fascia di produzione

Un solo passaggio su `predictions.csv`, con ogni evento come categoria: le
statistiche sono quindi direttamente confrontabili fra eventi e con lo strato
`normal` dell'intero anno.

`EVENT_SCOPE='days'` etichetta tutte le ore dei giorni toccati dall'evento. È
voluto: lo score del detector arriva in ritardo fino alla lunghezza della
finestra (60 ore), quindi restringersi alle sole ore dell'episodio taglierebbe
parte del giorno in cui il forecaster ha davvero sofferto. Con `'hours'` si
tengono solo le ore dell'episodio.

In [ ]:
EVENT_SCOPE = 'days'   # 'days' | 'hours'

event_labels = anomaly_extremes.event_timestamp_labels(events, scope=EVENT_SCOPE)
print(f'Ore etichettate: {len(event_labels):,} su {len(events)} eventi')
print('Intervallo etichette :', event_labels['timestamp'].min(), '->',
      event_labels['timestamp'].max())

# Controllo di aggancio: le etichette devono cadere nell'anno di test della run.
_head = pd.read_csv(out_dir / 'predictions.csv', usecols=['timestamp'], nrows=5000)
_span = pd.to_datetime(_head['timestamp'])
print('Prime previsioni     :', _span.min(), '->', _span.max())

event_comparison = anomaly_driver.build_anomaly_driver_comparison_figures(
    out_dir,
    event_labels,
    figure_subdir='extreme_events',
    metrics_name='extreme_event_metrics.csv',
    figures_per_category=True,
    chunksize=500_000,
)
event_metrics = event_comparison['metrics']
print('Categorie   :', event_comparison['categories'])
print('Righe per categoria:', event_comparison['row_counts'])
print('CSV metriche:', event_comparison['metrics_path'])
print('Cartelle per evento:')
for name, folder in event_comparison['category_dirs'].items():
    print(f'  {name}: {folder}')
display(event_metrics)

In [ ]:
for metric in ('mae', 'rmse', 'picp', 'nmpil', 'clc', 'bias'):
    print(f'\n=== {metric.upper()} ===')
    display(event_metrics.pivot(index='bin', columns='category', values=metric).round(3))

In [ ]:
# Degrado relativo rispetto allo strato normale, nello stesso bin.
normal = anomaly_driver.NORMAL_CATEGORY
wide = event_metrics.pivot(index='bin', columns='category', values='mae')
events_present = [c for c in wide.columns if c != normal]
delta = (wide[events_present].div(wide[normal], axis=0) - 1.0) * 100.0
display(delta.round(1))

# Copertura: scarto in punti percentuali rispetto al normale.
coverage = event_metrics.pivot(index='bin', columns='category', values='picp')
display(((coverage[events_present].sub(coverage[normal], axis=0)) * 100).round(2))

## 3. Figure

Per ogni fascia di produzione: boxplot Tukey esatti dell'errore assoluto e della
NMPIL per riga, istogramma dell'errore assoluto (upper 0.5% nell'ultimo bin) e
barre RMSE / PICP / CLC.

Due livelli, prodotti dalla stessa scansione:

- `figures/extreme_events/` — tutti gli eventi affiancati, per confrontarli fra loro;
- `figures/extreme_events/<evento>/` — **una cartella per evento**, con quell'evento
  accanto al solo strato `normal` e il suo `metrics.csv`. È il taglio leggibile
  quando l'evento va discusso da solo.

In [ ]:
print('Grafici totali:', len(event_comparison['figure_paths']))

# Figure combinate: tutti gli eventi affiancati.
for key in sorted(k for k in event_comparison['figure_paths'] if k.startswith('driver_compare')):
    print(key)
    display(Image(filename=str(event_comparison['figure_paths'][key])))

In [ ]:
# Figure di un singolo evento. Cambiare SHOW_EVENT per vedere gli altri.
SHOW_EVENT = events['event'].iloc[0] if not events.empty else None

folder = event_comparison['category_dirs'].get(SHOW_EVENT)
if folder is not None:
    print(SHOW_EVENT, '->', folder)
    display(pd.read_csv(folder / 'metrics.csv'))
    for png in sorted(folder.glob('*.png')):
        print(png.name)
        display(Image(filename=str(png)))
else:
    print('Evento senza figure:', SHOW_EVENT)

## 4. Classifica: raro non vuol dire difficile

Confronto diretto fra quanto il detector considera estremo un evento (score di
picco, finestre in coda) e quanto il forecaster ne soffre davvero (degrado di
MAE e caduta di copertura, pesati sul numero di righe).

In [ ]:
rows = []
for event in events_present:
    subset = event_metrics[event_metrics['category'] == event]
    reference = event_metrics[event_metrics['category'] == normal].set_index('bin')
    weights = subset['count'].to_numpy(float)
    if weights.sum() == 0:
        continue
    aligned = reference.loc[subset['bin']]
    rows.append({
        'evento': event,
        'n_righe': int(weights.sum()),
        'mae': float(np.average(subset['mae'], weights=weights)),
        'mae_normale': float(np.average(aligned['mae'], weights=weights)),
        'picp': float(np.average(subset['picp'], weights=weights)),
        'picp_normale': float(np.average(aligned['picp'], weights=weights)),
        'clc': float(np.average(subset['clc'], weights=weights)),
    })

ranking = pd.DataFrame(rows)
if ranking.empty:
    print('Nessun evento con righe diurne valide: classifica non calcolabile.')
else:
    ranking['mae_gap_%'] = ((ranking['mae'] / ranking['mae_normale'] - 1) * 100).round(1)
    ranking['picp_gap_pt'] = ((ranking['picp'] - ranking['picp_normale']) * 100).round(2)
    ranking = ranking.merge(
        events[['event', 'peak_score', 'n_extreme_windows']],
        left_on='evento', right_on='event', how='left',
    ).drop(columns='event').sort_values('mae_gap_%', ascending=False)
    display(ranking.round(3))

In [ ]:
# Il detector e il forecaster ordinano gli eventi allo stesso modo?
if not ranking.empty and len(ranking) > 2:
    correlation = ranking[
        ['peak_score', 'n_extreme_windows', 'mae_gap_%', 'picp_gap_pt']
    ].corr(method='spearman')
    display(correlation.round(3))
    print('Correlazione di rango detector vs forecaster:')
    print(f"  picco score   vs mae_gap: {correlation.loc['peak_score', 'mae_gap_%']:.3f}")
    print(f"  finestre coda vs mae_gap: {correlation.loc['n_extreme_windows', 'mae_gap_%']:.3f}")
else:
    print('Servono almeno tre eventi per una correlazione di rango.')

## 5. Riepilogo da copiare

In [ ]:
report_dump.dump_sections({
    'eventi': events.drop(columns='days') if not events.empty else None,
    'eventi_riassunto': event_summary if 'event_summary' in dir() else None,
    'righe_per_categoria': event_comparison['row_counts'],
    'metriche_evento': event_metrics,
    'delta_mae_vs_normale': delta if 'delta' in dir() else None,
    'classifica': ranking if 'ranking' in dir() and not ranking.empty else None,
}, max_rows=80)